# Análisis de Resultados UNI CEPRE

Este notebook permite analizar los resultados del examen de admisión UNI CEPRE.

## Instrucciones:
1. Sube el archivo `resultados_uni.csv` a este notebook
2. Ejecuta todas las celdas para ver los análisis

In [ ]:
# Importar librerías necesarias
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import files
import io

# Configurar estilo de visualización
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

# Configurar pandas para mostrar más columnas
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 50)

print("✅ Librerías importadas correctamente")

## 1. Cargar Datos

Sube el archivo CSV o proporciona la ruta si ya está en tu Drive

In [ ]:
# Opción 1: Subir archivo manualmente
uploaded = files.upload()

# Obtener el nombre del archivo
filename = list(uploaded.keys())[0]

# Leer el CSV
df = pd.read_csv(io.BytesIO(uploaded[filename]), encoding='utf-8')

print(f"✅ Archivo cargado: {filename}")
print(f"📊 Total de registros: {len(df)}")
print(f"📋 Columnas: {list(df.columns)}")

In [ ]:
# Opción 2: Si el archivo está en Google Drive
# Descomenta las siguientes líneas y ajusta la ruta

# from google.colab import drive
# drive.mount('/content/drive')
# df = pd.read_csv('/content/drive/MyDrive/ruta/a/resultados_uni.csv', encoding='utf-8')

## 2. Exploración Inicial de los Datos

In [ ]:
# Primeras filas
print("🔍 Primeras 10 filas:")
df.head(10)

In [ ]:
# Información general del dataset
print("📋 Información del DataFrame:")
df.info()
print("\n" + "="*60)
print("📊 Estadísticas descriptivas:")
df.describe()

In [ ]:
# Limpiar y preparar los datos
# Convertir puntaje_final a numérico
df['puntaje_final'] = pd.to_numeric(df['puntaje_final'], errors='coerce')

# Crear columna de ingresantes (puntaje > 0)
df['es_ingresante'] = df['puntaje_final'] > 0

# Verificar valores nulos
print("🔍 Valores nulos por columna:")
print(df.isnull().sum())
print("\n" + "="*60)
print(f"✅ Total de ingresantes: {df['es_ingresante'].sum()}")
print(f"📊 Total de registros: {len(df)}")

## 3. Estadísticas Generales

In [ ]:
# Estadísticas generales
total_registros = len(df)
total_ingresantes = df['es_ingresante'].sum()
puntaje_max = df['puntaje_final'].max()
puntaje_min = df['puntaje_final'].min()
puntaje_promedio = df['puntaje_final'].mean()
puntaje_mediana = df['puntaje_final'].median()

print("📊 ESTADÍSTICAS GENERALES")
print("="*60)
print(f"Total de registros: {total_registros:,}")
print(f"Total de ingresantes: {total_ingresantes:,}")
print(f"Tasa de ingreso: {(total_ingresantes/total_registros)*100:.2f}%")
print(f"\nPuntaje máximo: {puntaje_max:.2f}")
print(f"Puntaje mínimo: {puntaje_min:.2f}")
print(f"Puntaje promedio: {puntaje_promedio:.2f}")
print(f"Puntaje mediana: {puntaje_mediana:.2f}")
print(f"Desviación estándar: {df['puntaje_final'].std():.2f}")

## 4. Visualizaciones

In [ ]:
# Distribución de puntajes
plt.figure(figsize=(14, 6))

plt.subplot(1, 2, 1)
plt.hist(df['puntaje_final'].dropna(), bins=50, edgecolor='black', alpha=0.7, color='#e74c3c')
plt.axvline(df['puntaje_final'].mean(), color='green', linestyle='--', linewidth=2, label=f'Promedio: {df["puntaje_final"].mean():.2f}')
plt.axvline(df['puntaje_final'].median(), color='blue', linestyle='--', linewidth=2, label=f'Mediana: {df["puntaje_final"].median():.2f}')
plt.xlabel('Puntaje Final', fontsize=12)
plt.ylabel('Frecuencia', fontsize=12)
plt.title('Distribución de Puntajes Finales', fontsize=14, fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.boxplot(df['puntaje_final'].dropna(), vert=True)
plt.ylabel('Puntaje Final', fontsize=12)
plt.title('Boxplot de Puntajes Finales', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Estadísticas por modalidad
stats_modalidad = df.groupby('modalidad').agg({
    'codigo': 'count',
    'puntaje_final': ['mean', 'max', 'min', 'std'],
    'es_ingresante': 'sum'
}).round(2)

stats_modalidad.columns = ['Total', 'Promedio', 'Máximo', 'Mínimo', 'Desv. Est.', 'Ingresantes']
stats_modalidad = stats_modalidad.sort_values('Ingresantes', ascending=False)

print("📊 ESTADÍSTICAS POR MODALIDAD")
print("="*80)
print(stats_modalidad.to_string())

# Visualización
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

stats_modalidad['Ingresantes'].plot(kind='bar', ax=axes[0], color='#3498db')
axes[0].set_title('Número de Ingresantes por Modalidad', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Modalidad', fontsize=12)
axes[0].set_ylabel('Cantidad', fontsize=12)
axes[0].tick_params(axis='x', rotation=45)
axes[0].grid(True, alpha=0.3)

stats_modalidad['Promedio'].plot(kind='bar', ax=axes[1], color='#e74c3c')
axes[1].set_title('Puntaje Promedio por Modalidad', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Modalidad', fontsize=12)
axes[1].set_ylabel('Puntaje Promedio', fontsize=12)
axes[1].tick_params(axis='x', rotation=45)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Análisis por Especialidad

In [ ]:
# Estadísticas por especialidad
df_especialidad = df[df['especialidad'] != ''].copy()

stats_especialidad = df_especialidad.groupby('especialidad').agg({
    'codigo': 'count',
    'puntaje_final': ['mean', 'max', 'min', 'std'],
    'es_ingresante': 'sum'
}).round(2)

stats_especialidad.columns = ['Postulantes', 'Promedio', 'Máximo', 'Mínimo', 'Desv. Est.', 'Ingresantes']
stats_especialidad = stats_especialidad.sort_values('Ingresantes', ascending=False)

print("📊 ESTADÍSTICAS POR ESPECIALIDAD")
print("="*100)
print(stats_especialidad.to_string())

# Guardar en variable para uso posterior
top_especialidades = stats_especialidad.head(15)

In [ ]:
# Visualizaciones por especialidad
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Top 15 especialidades por número de ingresantes
top_especialidades['Ingresantes'].head(15).plot(kind='barh', ax=axes[0, 0], color='#3498db')
axes[0, 0].set_title('Top 15 Especialidades por Número de Ingresantes', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel('Cantidad de Ingresantes', fontsize=10)
axes[0, 0].grid(True, alpha=0.3)

# Top 15 especialidades por puntaje promedio
top_promedio = stats_especialidad.sort_values('Promedio', ascending=False).head(15)
top_promedio['Promedio'].plot(kind='barh', ax=axes[0, 1], color='#e74c3c')
axes[0, 1].set_title('Top 15 Especialidades por Puntaje Promedio', fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel('Puntaje Promedio', fontsize=10)
axes[0, 1].grid(True, alpha=0.3)

# Top 15 especialidades por puntaje máximo
top_maximo = stats_especialidad.sort_values('Máximo', ascending=False).head(15)
top_maximo['Máximo'].plot(kind='barh', ax=axes[1, 0], color='#2ecc71')
axes[1, 0].set_title('Top 15 Especialidades por Puntaje Máximo', fontsize=12, fontweight='bold')
axes[1, 0].set_xlabel('Puntaje Máximo', fontsize=10)
axes[1, 0].grid(True, alpha=0.3)

# Comparación: Ingresantes vs Promedio
axes[1, 1].scatter(stats_especialidad['Ingresantes'], stats_especialidad['Promedio'], 
                   s=100, alpha=0.6, color='#9b59b6')
axes[1, 1].set_xlabel('Número de Ingresantes', fontsize=10)
axes[1, 1].set_ylabel('Puntaje Promedio', fontsize=10)
axes[1, 1].set_title('Relación: Ingresantes vs Puntaje Promedio', fontsize=12, fontweight='bold')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Top Puntajes

In [ ]:
# Top 10 puntajes más altos
top_10 = df.nlargest(10, 'puntaje_final')[['codigo', 'nombre_completo', 'especialidad', 'modalidad', 'puntaje_final']]

print("🏆 TOP 10 PUNTAJES MÁS ALTOS")
print("="*100)
print(top_10.to_string(index=False))

# Visualización
plt.figure(figsize=(12, 8))
plt.barh(range(len(top_10)), top_10['puntaje_final'], color='#e74c3c')
plt.yticks(range(len(top_10)), [f"{row['codigo']} - {row['nombre_completo'][:30]}..." 
            for _, row in top_10.iterrows()])
plt.xlabel('Puntaje Final', fontsize=12)
plt.title('Top 10 Puntajes Más Altos', fontsize=14, fontweight='bold')
plt.gca().invert_yaxis()
plt.grid(True, alpha=0.3, axis='x')

# Agregar valores en las barras
for i, v in enumerate(top_10['puntaje_final']):
    plt.text(v + 10, i, f'{v:.2f}', va='center', fontsize=9)

plt.tight_layout()
plt.show()

## 7. Análisis por Rangos de Puntaje

In [ ]:
# Crear rangos de puntaje
def categorizar_puntaje(puntaje):
    if pd.isna(puntaje) or puntaje == 0:
        return 'Sin puntaje'
    elif puntaje >= 2000:
        return '2000+'
    elif puntaje >= 1500:
        return '1500-1999'
    elif puntaje >= 1000:
        return '1000-1499'
    else:
        return '<1000'

df['rango_puntaje'] = df['puntaje_final'].apply(categorizar_puntaje)

# Estadísticas por rango
rangos_stats = df.groupby('rango_puntaje').agg({
    'codigo': 'count',
    'puntaje_final': 'mean'
}).round(2)

rangos_stats.columns = ['Cantidad', 'Promedio']
rangos_stats = rangos_stats.sort_index()

print("📊 DISTRIBUCIÓN POR RANGOS DE PUNTAJE")
print("="*60)
print(rangos_stats.to_string())

# Visualización
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

rangos_stats['Cantidad'].plot(kind='bar', ax=axes[0], color='#3498db')
axes[0].set_title('Distribución por Rangos de Puntaje', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Rango de Puntaje', fontsize=12)
axes[0].set_ylabel('Cantidad de Estudiantes', fontsize=12)
axes[0].tick_params(axis='x', rotation=45)
axes[0].grid(True, alpha=0.3)

# Pie chart
rangos_con_puntaje = rangos_stats[rangos_stats.index != 'Sin puntaje']
axes[1].pie(rangos_con_puntaje['Cantidad'], labels=rangos_con_puntaje.index, autopct='%1.1f%%', 
           startangle=90, colors=['#e74c3c', '#3498db', '#2ecc71', '#f39c12'])
axes[1].set_title('Distribución Porcentual por Rangos', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

## 8. Búsqueda Personalizada

In [ ]:
# Buscar por código
codigo_buscar = "20490F"  # Cambia este código
resultado = df[df['codigo'] == codigo_buscar]

if len(resultado) > 0:
    print(f"✅ Resultado encontrado para código: {codigo_buscar}")
    print(resultado[['codigo', 'nombre_completo', 'modalidad', 'puntaje_final', 'especialidad']].to_string(index=False))
else:
    print(f"❌ No se encontró resultado para código: {codigo_buscar}")

In [ ]:
# Buscar por nombre (parcial)
nombre_buscar = "GARCIA"  # Cambia este nombre
resultados = df[df['nombre_completo'].str.contains(nombre_buscar, case=False, na=False)]

print(f"🔍 Resultados encontrados para '{nombre_buscar}': {len(resultados)}")
if len(resultados) > 0:
    print(resultados[['codigo', 'nombre_completo', 'puntaje_final', 'especialidad']].head(20).to_string(index=False))
else:
    print(f"❌ No se encontraron resultados para: {nombre_buscar}")

## 9. Exportar Resultados

In [ ]:
# Exportar estadísticas por especialidad a CSV
stats_especialidad.to_csv('estadisticas_por_especialidad.csv', encoding='utf-8-sig')
print("✅ Estadísticas por especialidad exportadas a 'estadisticas_por_especialidad.csv'")

# Exportar estadísticas por modalidad
stats_modalidad.to_csv('estadisticas_por_modalidad.csv', encoding='utf-8-sig')
print("✅ Estadísticas por modalidad exportadas a 'estadisticas_por_modalidad.csv'")

# Descargar archivos
files.download('estadisticas_por_especialidad.csv')
files.download('estadisticas_por_modalidad.csv')

## 10. Conclusiones y Resumen

### Resumen Ejecutivo:
- Total de registros analizados
- Tasa de ingreso general
- Especialidades más competitivas
- Distribución de puntajes

### Próximos Pasos:
- Análisis más detallado por especialidad
- Comparación entre modalidades
- Análisis temporal (si hay datos históricos)